In [ ]:
# Install dependencies 
%pip install anthropic python-dotenv

In [2]:
# Load env variables 
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create an API client 
from anthropic import Anthropic
from typing import Any

client = Anthropic()
model = "claude-sonnet-4-0"

In [4]:
# Helpers
def add_user_message(messages: list[dict[str,str]], text: str) -> None:
    """
    messages: conversation context.
    text: users message to add
    returns None

    Adds users message to the context window for claude.
    """
    user_message = {"role": "user", "coontent": text}
    messages.append(user_message)

def add_assistant_message(messages: list[str], text: str) -> None:
    """
    messages: conversation context.
    text: Claudes answer
    returns None

    Adds Claudes answer to the context window.
    """
    assitant_message = {"role": "assistant", "coontent": text}
    messages.append(assitant_message)

def chat(messages: list[dict[str,str]], system_prompt: str = None, tempreture: float = 1.0) -> str | Any:
    """
    messages: conversation context.
    system_prompt: system level prompt to give claude (acts as a golden rule) default to None
    tempreture: influences the determination of the model 
    returns Claudes answer

    Performs a request to claude wuth the entire context window ad returns the answer
    """
    params = {
        "model": model, 
        "max_tokens": 1000, 
        "messages": messages,
        "tempreture": tempreture
        }

    if system_prompt: 
        params["system"] = system_prompt

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
# response streaming lets users see text appear chunk by chunk 
# as Claude generates it, creating a much more responsive feel.

# Stream event types 
# MessageStart - A new message is being sent
# ContentBlockStart - Start of a new block containing text, tool use, or other content
# ContentBlockDelta - Chunks of the actual generated text
# ContentBlockStop - The current content block has been completed
# MessageDelta - The current message is complete
# MessageStop - End of information about the current message

messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database.")

# returns an iterator for manually parsing events 
stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)


for event in stream:
    print(event)

In [ ]:
messages = []

add_user_message(messages, "Write a 1 sentence description of a fake database.")

# simplify with the SDK
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")

# Get the complete message for database storage
final_message = stream.get_final_message()